# Eksploracyjna Analiza Danych (EDA) — Black Friday

Zestaw danych: [Black Friday (Kaggle)](https://www.kaggle.com/sdolezel/black-friday?select=train.csv)

**Spis treści**
1. Importowanie bibliotek
2. Ładowanie zestawu danych
3. Eksploracja danych strukturalnych
4. Obsługa duplikatów
5. Obsługa wartości odstających
6. Obsługa brakujących wartości
7. Analiza jednowymiarowa
8. Analiza dwuwymiarowa

## 1. Importowanie bibliotek języka Python

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

## 2. Ładowanie zestawu danych w języku Python

In [ ]:
df = pd.read_csv('train.csv')
df

## 3. Eksploracja danych strukturalnych

Pierwszy krok EDA — zrozumienie metadanych (Metadata).

### Pierwsze 5 obserwacji

In [ ]:
df.head()

### Ostatnie 5 obserwacji

In [ ]:
df.tail()

### Liczba zmiennych i obserwacji

In [ ]:
df.shape

In [ ]:
print('Liczba obserwacji:', df.shape[0])
print('Liczba zmiennych:', df.shape[1])

### Nazwy zmiennych i typy danych

In [ ]:
df.dtypes

### Liczba niebrakujących wartości dla każdej zmiennej

In [ ]:
df.count()

### Statystyki opisowe

In [ ]:
pd.set_option('display.max_columns', None)
df.describe()

In [ ]:
df.describe(include='all')

### Pełne metadane zestawu danych

In [ ]:
df.info()

## 4. Obsługa duplikatów

Dwa kroki: **wykrywanie** i **usuwanie** duplikatów.

In [ ]:
df.duplicated()

In [ ]:
print('Liczba zduplikowanych wierszy:', df.duplicated().sum())

Brak duplikatów całych obserwacji — każda obserwacja jest unikalna.

Usunięcie duplikatów (gdyby występowały):
```python
df.drop_duplicates()
```

W zmiennej `User_ID` wartości się powtarzają (ten sam użytkownik kupuje wiele produktów). Poniżej — podgląd bez trwałej zmiany danych:

In [ ]:
df.drop_duplicates(subset='User_ID', inplace=False)

## 5. Obsługa wartości odstających

Wartości odstające (outliers) to skrajne wartości na dolnej i górnej skali danych.

Metody wykrywania: percentylowa, **IQR**, odchylenie standardowe.

Rozważamy zmienną `Purchase` — metoda IQR.

In [ ]:
p0 = df.Purchase.min()
p100 = df.Purchase.max()
q1 = df.Purchase.quantile(0.25)
q2 = df.Purchase.quantile(0.5)
q3 = df.Purchase.quantile(0.75)
iqr = q3 - q1

lc = q1 - 1.5 * iqr
uc = q3 + 1.5 * iqr

print('p0 =', p0, ', p100 =', p100, ', lc =', lc, ', uc =', uc)
print('q1 =', q1, ', q2 =', q2, ', q3 =', q3, ', iqr =', iqr)

**Konwencja:**
- jeśli `lc < p0` → brak wartości odstających od dołu
- jeśli `uc > p100` → brak wartości odstających od góry

Oczekiwanie: `lc < p0` (brak outlierów od dołu), ale `uc < p100` (są outliery od góry).

In [ ]:
print('Outliery od dołu:', lc >= p0)
print('Outliery od góry:', uc < p100)

In [ ]:
df.Purchase.plot(kind='box')
plt.title('Purchase — przed przycięciem outlierów')
plt.show()

### Leczenie wartości odstających (przycinanie / clipping)

Nie usuwamy obserwacji — przycinamy wartości powyżej górnego odcięcia do `uc`.

In [ ]:
df['Purchase'] = df['Purchase'].clip(upper=uc)
df.Purchase.plot(kind='box')
plt.title('Purchase — po przycięciu outlierów')
plt.show()

## 6. Obsługa wartości brakujących

Dwa kroki: **wykrywanie** i **leczenie**.

In [ ]:
df.isna()

### Procent brakujących wartości w każdej zmiennej

In [ ]:
missing_pct = (df.isna().sum() / df.shape[0] * 100).sort_values(ascending=False)
missing_pct

Braki występują w `Product_Category_2` (~31,5%) oraz `Product_Category_3` (~69,7%).

### Leczenie brakujących wartości

| Metoda | Kiedy |
|--------|--------|
| Upuść zmienną | bardzo duży % braków |
| Upuść obserwacje | mały % braków |
| Imputacja | umiarkowany % braków |

- `Product_Category_2` (~31%) → imputacja **modą** (zmienna kategoryczna)
- `Product_Category_3` (~70%) → **porzucenie zmiennej**

In [ ]:
mode_pc2 = df.Product_Category_2.mode()[0]
print('Tryb Product_Category_2:', mode_pc2)

df['Product_Category_2'] = df['Product_Category_2'].fillna(mode_pc2)
df.isna().sum()

In [ ]:
# Porzucenie kolumn z brakami (tu: Product_Category_3)
df.dropna(axis=1, inplace=True)
df.dtypes

In [ ]:
print('Pozostałe zmienne:', list(df.columns))
print('Braki po leczeniu:', df.isna().sum().sum())

## 7. Analiza jednowymiarowa

Wykresy dla jednej zmiennej — rozkład (ciągłe) lub skład/częstotliwość (kategoryczne).

### Zmienne ciągłe — `Purchase`

In [ ]:
df.Purchase.hist()
plt.title('Histogram — Purchase')
plt.show()

In [ ]:
df.Purchase.plot(kind='hist', grid=True)
plt.title('Histogram (plot) — Purchase')
plt.show()

In [ ]:
plt.hist(df.Purchase)
plt.grid(True)
plt.title('Histogram (matplotlib) — Purchase')
plt.show()

In [ ]:
df.Purchase.plot(kind='box')
plt.title('Boxplot — Purchase')
plt.show()

In [ ]:
plt.boxplot(df.Purchase)
plt.title('Boxplot (matplotlib) — Purchase')
plt.show()

### Zmienne kategoryczne

In [ ]:
# Skład płci
df.groupby('Gender').City_Category.count().plot(kind='pie', autopct='%1.1f%%')
plt.title('Skład — Gender')
plt.ylabel('')
plt.show()

In [ ]:
# Dystrybucja stanu cywilnego
sns.countplot(x=df.Marital_Status)
plt.title('Countplot — Marital_Status')
plt.show()

In [ ]:
# Skład — kategoria miasta
df.groupby('City_Category').City_Category.count().plot(kind='pie', autopct='%1.1f%%')
plt.title('Skład — City_Category')
plt.ylabel('')
plt.show()

In [ ]:
# Rozkład wieku
sns.countplot(x=df.Age)
plt.title('Countplot — Age')
plt.show()

In [ ]:
# Przebywanie w aktualnym miejscu zamieszkania
df.groupby('Stay_In_Current_City_Years').City_Category.count().plot(kind='pie', autopct='%1.1f%%')
plt.title('Skład — Stay_In_Current_City_Years')
plt.ylabel('')
plt.show()

In [ ]:
# Podział zawodowy
sns.countplot(x=df.Occupation)
plt.title('Countplot — Occupation')
plt.show()

In [ ]:
# Dystrybucja Product_Category_1
df.groupby('Product_Category_1').City_Category.count().plot(kind='barh')
plt.title('Rozkład — Product_Category_1')
plt.show()

## 8. Analiza dwuwymiarowa

Trzy przypadki: numeryczne–numeryczne, numeryczne–kategoryczne, kategoryczne–kategoryczne.

### Numeryczne i numeryczne

In [ ]:
# Wykres punktowy (hipotetycznie traktujemy kategorie produktów jako numeryczne)
df.plot(x='Product_Category_1', y='Product_Category_2', kind='scatter', alpha=0.3)
plt.title('Scatter — Product_Category_1 vs Product_Category_2')
plt.show()

In [ ]:
plt.scatter(x=df.Product_Category_1, y=df.Product_Category_2, alpha=0.3)
plt.xlabel('Product_Category_1')
plt.ylabel('Product_Category_2')
plt.title('Scatter (matplotlib)')
plt.show()

In [ ]:
# Macierz korelacji
corr = df.select_dtypes(['float64', 'int64']).corr()
corr

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(['float64', 'int64']).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Heatmap — korelacja zmiennych numerycznych')
plt.show()

### Numeryczne i kategoryczne

In [ ]:
# Purchase vs Occupation — wykres słupkowy
df.groupby('Occupation').Purchase.sum().plot(kind='bar')
plt.title('Suma Purchase wg Occupation')
plt.show()

In [ ]:
summary = df.groupby('Occupation').Purchase.sum()
plt.bar(x=summary.index, height=summary.values)
plt.title('Suma Purchase wg Occupation (matplotlib)')
plt.show()

In [ ]:
sns.barplot(x=summary.index, y=summary.values)
plt.title('Suma Purchase wg Occupation (seaborn)')
plt.show()

In [ ]:
# Purchase vs Age — wykres liniowy
df.groupby('Age').Purchase.sum().plot(kind='line', marker='o')
plt.title('Suma Purchase wg Age')
plt.show()

In [ ]:
# Skład Purchase wg Gender — pie
df.groupby('Gender').Purchase.sum().plot(kind='pie', autopct='%1.1f%%')
plt.title('Skład Purchase wg Gender')
plt.ylabel('')
plt.show()

In [ ]:
# Purchase vs City_Category — area
df.groupby('City_Category').Purchase.sum().plot(kind='area')
plt.title('Suma Purchase wg City_Category')
plt.show()

In [ ]:
# Purchase vs Stay_In_Current_City_Years — barh
df.groupby('Stay_In_Current_City_Years').Purchase.sum().plot(kind='barh')
plt.title('Suma Purchase wg Stay_In_Current_City_Years')
plt.show()

In [ ]:
# Purchase vs Marital_Status — boxplot
sns.boxplot(x='Marital_Status', y='Purchase', data=df)
plt.title('Purchase wg Marital_Status')
plt.show()

### Kategoryczne i kategoryczne

In [ ]:
# Związek Age × Gender
pd.crosstab(df.Age, df.Gender)

In [ ]:
sns.heatmap(pd.crosstab(df.Age, df.Gender), annot=True, fmt='d')
plt.title('Heatmap — Age × Gender')
plt.show()

In [ ]:
# Związek City_Category × Stay_In_Current_City_Years
sns.heatmap(pd.crosstab(df.City_Category, df.Stay_In_Current_City_Years), annot=True, fmt='d')
plt.title('Heatmap — City_Category × Stay_In_Current_City_Years')
plt.show()

---
## Podsumowanie EDA

1. **Struktura:** zestaw Black Friday — transakcje zakupowe z atrybutami użytkownika i produktu.
2. **Duplikaty:** brak zduplikowanych wierszy; `User_ID` naturalnie się powtarza.
3. **Outliery:** w `Purchase` wykryto wartości odstające od góry (IQR) — przycięto do `uc`.
4. **Braki:** `Product_Category_2` uzupełniono modą; `Product_Category_3` usunięto (>50% braków).
5. **Jedno-/dwuwymiarowa:** rozkłady zmiennych oraz relacje Purchase z Occupacją, wiekiem, płcią, miastem itd.